# Compustat-CRSP Merge: EDA Version (Debugging/Exploration)

This notebook focuses on exploratory data analysis to understand the data
flow and identify where errors might be occurring. Minimal functions,
maximum visibility.

**Run this to understand:**
- What data is being loaded at each step
- Data types and shapes
- Where data might be getting lost
- Sample rows at each transformation
- Intermediate results before merges

## Setup: Parameters and Helper Functions

In [8]:
"""
============================================================================
Compustat-CRSP Merge: EDA Version (Debugging/Exploration)
============================================================================

This script focuses on exploratory data analysis to understand the data
flow and identify where errors might be occurring. Minimal functions,
maximum visibility.

Run this to understand:
- What data is being loaded at each step
- Data types and shapes
- Where data might be getting lost
- Sample rows at each transformation
- Intermediate results before merges
============================================================================
"""

import os
import sys
import pickle
import warnings
from datetime import datetime, date
from pathlib import Path

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine
import psycopg2

warnings.filterwarnings('ignore')

# Try to import optional packages
try:
    import wrds
    HAS_WRDS = True
except ImportError:
    HAS_WRDS = False
    print("Warning: wrds package not found. Install with: pip install wrds")

# ============================================================================
# MINIMAL HELPER FUNCTIONS
# ============================================================================

def connect_wharton(username="zkg232"):
    """Connect to WRDS PostgreSQL database."""
    if HAS_WRDS:
        return wrds.Connection(wrds_username=username)
    else:
        conn_string = (
            f"postgresql://{username}@wrds-pgdata.wharton.upenn.edu:9737/wrds"
            "?sslmode=require"
        )
        return create_engine(conn_string)

def execute_sql(query, con):
    """Execute SQL query and return DataFrame. Handles both WRDS and SQLAlchemy connections."""
    if HAS_WRDS and isinstance(con, wrds.Connection):
        return con.raw_sql(query)
    else:
        return pd.read_sql(query, con)

def print_section(title):
    """Print a section header."""
    print("\n" + "="*80)
    print(f"  {title}")
    print("="*80 + "\n")

def inspect_dataframe(df, name, show_sample=True, n_sample=5):
    """Inspect a dataframe with detailed information."""
    print(f"\n--- {name} ---")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\nColumn names: {list(df.columns)}")
    print(f"\nData types:")
    print(df.dtypes)
    print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Check for missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"\nMissing values:")
        print(missing[missing > 0])
    else:
        print("\nNo missing values")
    
    # Show sample rows
    if show_sample and len(df) > 0:
        print(f"\nFirst {n_sample} rows:")
        print(df.head(n_sample))
    
    # Show date ranges if date column exists
    date_cols = [col for col in df.columns if 'date' in col.lower()]
    for col in date_cols:
        if col in df.columns and df[col].dtype in ['datetime64[ns]', 'object']:
            try:
                dates = pd.to_datetime(df[col], errors='coerce')
                valid_dates = dates.dropna()
                if len(valid_dates) > 0:
                    print(f"\n{col} range: {valid_dates.min()} to {valid_dates.max()}")
            except:
                pass
    
    print("\n" + "-"*80)




## Setup: Parameters (match `reprex_final.py`)

Adjust these to control the sample. Keeping these aligned makes the EDA notebook directly comparable to the functionalized script.

In [ ]:
print_section("SETUP: PARAMETERS")

# Keep these consistent with `notes/reprex_final.py` defaults
wrds_username = "zkg232"

# Defaults match the functionalized script. For focused debugging, set:
#   cusip_target = test_cusip
# (use 8-digit CUSIP)
test_cusip = "93114210"
cusip_target = None

start_year = 1980
end_year = 2024
start_date = pd.Timestamp(f"{start_year}-01-01")
end_date = pd.Timestamp(f"{end_year}-12-31")

print(f"wrds_username: {wrds_username}")
print(f"test_cusip: {test_cusip}")
print(f"cusip_target: {cusip_target}")
print(f"start_year/end_year: {start_year} / {end_year}")
print(f"start_date/end_date: {start_date.date()} / {end_date.date()}")


## Step 0: Connect to WRDS

This creates the `con` object used by all subsequent SQL queries.

In [9]:
print_section("STEP 0: Connect to WRDS")

# This creates the connection object used throughout the notebook.
# If you are prompted for a WRDS password, enter it in the prompt.
_username = wrds_username if 'wrds_username' in globals() else "zkg232"
con = connect_wharton(username=_username)

print(f"Connection type: {type(con)}")



  STEP 0: Connect to WRDS

Loading library list...
Done
Connection type: <class 'wrds.sql.Connection'>


## Step 1: CCM Linking Table


In [10]:
# ============================================================================

print_section("STEP 1: CCM Linking Table")

# If you run this cell out of order, ensure we have a WRDS connection
if "con" not in globals() or con is None:
    _username = wrds_username if 'wrds_username' in globals() else "zkg232"
    con = connect_wharton(username=_username)

# Ensure parameters exist (keep aligned with `reprex_final.py` defaults)
if 'start_year' not in globals():
    start_year = 1980
if 'end_year' not in globals():
    end_year = 2024
if 'start_date' not in globals():
    start_date = pd.Timestamp('1980-01-01')
if 'end_date' not in globals():
    end_date = pd.Timestamp('2024-12-31')
if 'cusip_target' not in globals():
    cusip_target = None

# Optional: restrict to a test CUSIP (fast EDA)
cusip_gvkey = None
if cusip_target is not None:
    cusip_target_8 = str(cusip_target)[:8]
    print(f"Filtering to test CUSIP (8-digit): {cusip_target_8}")
    cusip_query = f"""
    SELECT DISTINCT gvkey
    FROM comp.names
    WHERE SUBSTR(cusip, 1, 8) = '{cusip_target_8}'
    """
    cusip_gvkey = execute_sql(cusip_query, con)
    inspect_dataframe(cusip_gvkey, f"CUSIP→GVKEY ({cusip_target_8})", show_sample=True)

ccm_query = """
SELECT gvkey, lpermno as permno, lpermco as permco,
       linkdt, COALESCE(linkenddt, '9999-12-31'::date) as linkenddt
FROM crsp.ccmxpf_lnkhist
WHERE linktype IN ('LU', 'LC') AND linkprim IN ('P', 'C')
"""

# Apply gvkey filter if a CUSIP was provided
if cusip_gvkey is not None and len(cusip_gvkey) > 0:
    gvkeys = "', '".join(cusip_gvkey['gvkey'].astype(str))
    ccm_query += f" AND gvkey IN ('{gvkeys}')\n"

print("Query:")
print(ccm_query[:200] + "..." if len(ccm_query) > 200 else ccm_query)

ccm_link = execute_sql(ccm_query, con)
inspect_dataframe(ccm_link, "CCM Link (raw)")

# Convert dates
print("\nConverting dates...")
ccm_link['linkdt'] = pd.to_datetime(ccm_link['linkdt'], errors='coerce')
ccm_link['linkenddt'] = pd.to_datetime(ccm_link['linkenddt'], errors='coerce')

# Check for problematic dates
print(f"\nLinkenddt with '9999-12-31': {(ccm_link['linkenddt'] > pd.Timestamp('2099-12-31')).sum()} rows")
max_date = pd.Timestamp('2099-12-31')
ccm_link['linkenddt'] = ccm_link['linkenddt'].where(
    ccm_link['linkenddt'] <= max_date, 
    pd.NaT
)

inspect_dataframe(ccm_link, "CCM Link (after date conversion)")



  STEP 1: CCM Linking Table

Query:

SELECT gvkey, lpermno as permno, lpermco as permco,
       linkdt, COALESCE(linkenddt, '9999-12-31'::date) as linkenddt
FROM crsp.ccmxpf_lnkhist
WHERE linktype IN ('LU', 'LC') AND linkprim IN ('P', '...

--- CCM Link (raw) ---
Shape: 32,849 rows × 5 columns

Column names: ['gvkey', 'permno', 'permco', 'linkdt', 'linkenddt']

Data types:
gvkey        string[python]
permno              Float64
permco              Float64
linkdt       string[python]
linkenddt    string[python]
dtype: object

Memory usage: 5.98 MB

No missing values

First 5 rows:
    gvkey   permno   permco      linkdt   linkenddt
0  001000  25881.0  23369.0  1970-11-13  1978-06-30
1  001001  10015.0   6398.0  1983-09-20  1986-07-31
2  001002  10023.0  22159.0  1972-12-14  1973-06-05
3  001003  10031.0   6672.0  1983-12-07  1989-08-16
4  001004  54594.0  20000.0  1972-04-24  9999-12-31

--------------------------------------------------------------------------------

Converting dates.

## Step 2: Compustat Fundamentals


In [11]:
# ============================================================================

print_section("STEP 2: Compustat Fundamentals")

# If you run this cell out of order, ensure we have a WRDS connection
if "con" not in globals() or con is None:
    _username = wrds_username if 'wrds_username' in globals() else "zkg232"
    con = connect_wharton(username=_username)

# Ensure parameters exist (keep aligned with `reprex_final.py` defaults)
if 'start_year' not in globals():
    start_year = 1980
if 'end_year' not in globals():
    end_year = 2024
if 'start_date' not in globals():
    start_date = pd.Timestamp('1980-01-01')
if 'end_date' not in globals():
    end_date = pd.Timestamp('2024-12-31')
if 'cusip_target' not in globals():
    cusip_target = None

# If Step 1 hasn't run, we may not have cusip_gvkey yet
cusip_gvkey = globals().get('cusip_gvkey', None)
if cusip_target is not None and (cusip_gvkey is None):
    cusip_target_8 = str(cusip_target)[:8]
    cusip_query = f"""
    SELECT DISTINCT gvkey
    FROM comp.names
    WHERE SUBSTR(cusip, 1, 8) = '{cusip_target_8}'
    """
    cusip_gvkey = execute_sql(cusip_query, con)

funda_query = f"""
SELECT f.gvkey, f.datadate, f.fyear, f.fyr, f.apdedate, f.fdate, f.pdate,
       f.conm, f.tic, f.cusip, f.cik, f.ni, f.xrd,
       SUBSTR(n.gind, 1, 2) as gsector,
       f.datadate as endfyr,
       f.datadate - INTERVAL '11 months' as begfyr,
       f.datadate + INTERVAL '3 months' as available_date
FROM comp.funda f
LEFT JOIN comp.names n ON f.gvkey = n.gvkey
WHERE f.fyear >= {start_year - 10} AND f.fyear <= {end_year}
  AND f.indfmt = 'INDL' AND f.datafmt = 'STD' 
  AND f.popsrc = 'D' AND f.consol = 'C'
  AND f.curcd = 'USD' AND f.fic = 'USA'
  AND f.exchg BETWEEN 11 AND 19
  AND (f.sich IS NULL OR (f.sich NOT BETWEEN 6000 AND 6999 AND f.sich != 2834))
"""

# Apply gvkey filter if a CUSIP was provided
if cusip_gvkey is not None and len(cusip_gvkey) > 0:
    gvkeys = "', '".join(cusip_gvkey['gvkey'].astype(str))
    funda_query += f" AND f.gvkey IN ('{gvkeys}')\n"

print("Query:")
print(funda_query[:300] + "..." if len(funda_query) > 300 else funda_query)

funda = execute_sql(funda_query, con)
inspect_dataframe(funda, "Compustat Fundamentals (raw)")

# Convert dates
print("\nConverting dates...")
funda['datadate'] = pd.to_datetime(funda['datadate'], errors='coerce')
funda['endfyr'] = pd.to_datetime(funda['endfyr'], errors='coerce')
funda['begfyr'] = pd.to_datetime(funda['begfyr'], errors='coerce')
funda['available_date'] = pd.to_datetime(funda['available_date'], errors='coerce')

# Check date validity
print(f"\nRows with invalid dates:")
print(f"  datadate: {funda['datadate'].isna().sum()}")
print(f"  endfyr: {funda['endfyr'].isna().sum()}")
print(f"  begfyr: {funda['begfyr'].isna().sum()}")
print(f"  available_date: {funda['available_date'].isna().sum()}")

# Drop rows with invalid dates
funda_before = len(funda)
funda = funda[
    funda['datadate'].notna() & 
    funda['endfyr'].notna() & 
    funda['begfyr'].notna() & 
    funda['available_date'].notna()
]
funda_after = len(funda)
print(f"\nDropped {funda_before - funda_after} rows with invalid dates ({funda_before:,} → {funda_after:,})")

inspect_dataframe(funda, "Compustat Fundamentals (after date filtering)")

# Check available_date filter
print(f"\nRows where available_date <= end_date ({end_date}): {(funda['available_date'] <= end_date).sum():,}")
funda_filtered = funda[funda['available_date'] <= end_date]
print(f"After filtering by available_date: {len(funda_filtered):,} rows")



  STEP 2: Compustat Fundamentals



: 

## Step 3: CRSP Monthly Stock File


In [ ]:
# ============================================================================

print_section("STEP 3: CRSP Monthly Stock File")

# If you run this cell out of order, ensure we have a WRDS connection
if "con" not in globals() or con is None:
    _username = wrds_username if 'wrds_username' in globals() else "zkg232"
    con = connect_wharton(username=_username)

# Ensure parameters exist
if 'start_date' not in globals():
    start_date = pd.Timestamp('1980-01-01')
if 'end_date' not in globals():
    end_date = pd.Timestamp('2024-12-31')

msf_query = f"""
SELECT m.permno, m.date, m.ret, m.prc, m.vol, m.shrout,
       ABS(m.prc * m.shrout) as mktcap
FROM crsp.msf m
INNER JOIN crsp.msenames n ON m.permno = n.permno
WHERE m.date >= '{start_date.date()}' AND m.date <= '{end_date.date()}'
  AND m.ret IS NOT NULL AND m.prc IS NOT NULL AND m.shrout IS NOT NULL
  AND m.date >= n.namedt AND m.date <= COALESCE(n.nameendt, '9999-12-31'::date)
  AND (n.siccd NOT BETWEEN 6000 AND 6999 AND n.siccd != 2834)
  AND n.shrcd IN (10, 11) AND n.exchcd IN (1, 2, 3)
"""

# Optional: if Step 1 filtered CCM to a single CUSIP, restrict MSF to those PERMNOs
ccm_link_local = globals().get('ccm_link', None)
if ccm_link_local is not None and 'cusip_target' in globals() and cusip_target is not None:
    permnos = (
        ccm_link_local['permno']
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    if len(permnos) > 0 and len(permnos) <= 500:
        permno_list = ", ".join(str(p) for p in sorted(permnos))
        msf_query += f"  AND m.permno IN ({permno_list})\n"
        print(f"Restricting MSF to {len(permnos)} PERMNO(s) from CCM (CUSIP-filtered)")

print("Query:")
print(msf_query[:300] + "..." if len(msf_query) > 300 else msf_query)

msf = execute_sql(msf_query, con)
inspect_dataframe(msf, "CRSP MSF (raw)")

# Convert dates
print("\nConverting dates...")
msf['date'] = pd.to_datetime(msf['date'], errors='coerce')
msf = msf[msf['date'].notna()]

# Check data types
print("\nChecking numeric columns...")
print(f"  ret: {msf['ret'].dtype}, nulls: {msf['ret'].isna().sum()}")
print(f"  prc: {msf['prc'].dtype}, nulls: {msf['prc'].isna().sum()}")
print(f"  shrout: {msf['shrout'].dtype}, nulls: {msf['shrout'].isna().sum()}")
print(f"  mktcap: {msf['mktcap'].dtype}, nulls: {msf['mktcap'].isna().sum()}")

# Convert to numeric
msf['ret'] = pd.to_numeric(msf['ret'], errors='coerce')
msf['prc'] = pd.to_numeric(msf['prc'], errors='coerce')
msf['shrout'] = pd.to_numeric(msf['shrout'], errors='coerce')
msf['mktcap'] = pd.to_numeric(msf['mktcap'], errors='coerce')

inspect_dataframe(msf, "CRSP MSF (after conversion)")


## Step 4: Delisting Returns


In [ ]:
# ============================================================================

print_section("STEP 4: Delisting Returns")

delist_query = f"""
SELECT permno, dlstdt as date, dlret
FROM crsp.msedelist
WHERE dlstdt >= '{start_date.date()}' AND dlstdt <= '{end_date.date()}'
  AND dlret IS NOT NULL
"""

print("Query:")
print(delist_query)

delist = execute_sql(delist_query, con)
inspect_dataframe(delist, "Delisting Returns (raw)")

# Convert dates and numeric
delist['date'] = pd.to_datetime(delist['date'], errors='coerce')
delist['dlret'] = pd.to_numeric(delist['dlret'], errors='coerce')
delist = delist[delist['date'].notna() & delist['dlret'].notna()]

inspect_dataframe(delist, "Delisting Returns (after conversion)")


## Step 5: Apply CCM Linking


In [ ]:
# ============================================================================

print_section("STEP 5: Apply CCM Linking to CRSP Data")

print(f"Before merge:")
print(f"  msf: {len(msf):,} rows")
print(f"  ccm_link: {len(ccm_link):,} rows")

# Merge CCM link with MSF
link_msf = msf.merge(ccm_link, on='permno', how='inner')
print(f"\nAfter merge on permno: {len(link_msf):,} rows")

# Filter by link dates
print(f"\nFiltering by link dates...")
print(f"  Rows where date >= linkdt: {(link_msf['date'] >= link_msf['linkdt']).sum():,}")
print(f"  Rows where linkenddt is NA (open-ended): {link_msf['linkenddt'].isna().sum():,}")
print(f"  Rows where date <= linkenddt (when not NA): {(link_msf['linkenddt'].notna() & (link_msf['date'] <= link_msf['linkenddt'])).sum():,}")

link_msf = link_msf[
    (link_msf['date'] >= link_msf['linkdt']) &
    (link_msf['linkenddt'].isna() | (link_msf['date'] <= link_msf['linkenddt']))
]
print(f"After date filtering: {len(link_msf):,} rows")

# Handle duplicates
print(f"\nChecking for duplicates...")
duplicates = link_msf.duplicated(subset=['permno', 'date'], keep=False)
print(f"  Duplicate (permno, date) pairs: {duplicates.sum():,}")

link_msf = link_msf.sort_values(['permno', 'date', 'linkdt'])
link_msf = link_msf.drop_duplicates(subset=['permno', 'date'], keep='last')
print(f"After deduplication: {len(link_msf):,} rows")

inspect_dataframe(link_msf, "CRSP with CCM Link (after filtering)")


## Step 6: Merge CRSP Dates with Fundamentals


In [ ]:
# ============================================================================

print_section("STEP 6: Merge CRSP Dates with Fundamentals")

# Get unique (gvkey, permno, date) combinations
crsp_dates = link_msf[['gvkey', 'permno', 'date']].drop_duplicates()
print(f"Unique (gvkey, permno, date) combinations: {len(crsp_dates):,}")

print(f"\nBefore merge:")
print(f"  crsp_dates: {len(crsp_dates):,} rows")
print(f"  funda_filtered: {len(funda_filtered):,} rows")

# Merge
merged = crsp_dates.merge(
    funda_filtered, on='gvkey', how='inner', suffixes=('', '_funda')
)
print(f"After merge on gvkey: {len(merged):,} rows")

# Filter by available_date
print(f"\nFiltering by available_date...")
print(f"  Rows where date >= available_date: {(merged['date'] >= merged['available_date']).sum():,}")
merged = merged[merged['date'] >= merged['available_date']]
print(f"After filtering: {len(merged):,} rows")

# Handle duplicates
print(f"\nChecking for duplicates...")
duplicates = merged.duplicated(subset=['gvkey', 'permno', 'date'], keep=False)
print(f"  Duplicate (gvkey, permno, date) pairs: {duplicates.sum():,}")

merged = merged.sort_values(['gvkey', 'permno', 'date', 'datadate'])
merged = merged.drop_duplicates(subset=['gvkey', 'permno', 'date'], keep='last')
print(f"After deduplication: {len(merged):,} rows")

inspect_dataframe(merged, "Merged CRSP Dates + Fundamentals")


## Step 7: Join CRSP Data Back


In [ ]:
# ============================================================================

print_section("STEP 7: Join CRSP Data Back")

crsp_data = link_msf[['gvkey', 'permno', 'date', 'ret', 'prc', 'vol', 'mktcap']].copy()
crsp_data = crsp_data.rename(columns={'ret': 'ret_crsp', 'prc': 'prc_crsp'})
crsp_data['prc_crsp'] = crsp_data['prc_crsp'].abs()
crsp_data['ret_crsp'] = pd.to_numeric(crsp_data['ret_crsp'], errors='coerce')

print(f"Before merge:")
print(f"  merged: {len(merged):,} rows")
print(f"  crsp_data: {len(crsp_data):,} rows")

merged = merged.merge(crsp_data, on=['gvkey', 'permno', 'date'], how='inner')
print(f"After merge: {len(merged):,} rows")

inspect_dataframe(merged, "Merged with CRSP Data")


## Step 8: Add Delisting Returns


In [ ]:
# ============================================================================

print_section("STEP 8: Add Delisting Returns")

print(f"Before merge:")
print(f"  merged: {len(merged):,} rows")
print(f"  delist: {len(delist):,} rows")

merged = merged.merge(delist, on=['permno', 'date'], how='left')
print(f"After merge: {len(merged):,} rows")
print(f"  Rows with delisting returns: {merged['dlret'].notna().sum():,}")

inspect_dataframe(merged, "Merged with Delisting Returns")


## Step 9: Calculate Effective Returns


In [ ]:
# ============================================================================

print_section("STEP 9: Calculate Effective Returns")

print("Calculating effective returns...")
print(f"  Rows with ret_crsp: {merged['ret_crsp'].notna().sum():,}")
print(f"  Rows with dlret: {merged['dlret'].notna().sum():,}")

merged['ret_eff'] = merged['ret_crsp'].fillna(0) + merged['dlret'].fillna(0)

print(f"\nEffective returns:")
print(f"  Non-null: {merged['ret_eff'].notna().sum():,}")
print(f"  Mean: {merged['ret_eff'].mean():.6f}")
print(f"  Std: {merged['ret_eff'].std():.6f}")
print(f"  Min: {merged['ret_eff'].min():.6f}")
print(f"  Max: {merged['ret_eff'].max():.6f}")


## Step 10: Add GICS Sector Names


In [ ]:
# ============================================================================

print_section("STEP 10: Add GICS Sector Names")

gics_map = pd.DataFrame({
    'gsector': ['10', '15', '20', '25', '30', '35', '40', '45', '50', '55', '60'],
    'sector_name': [
        'Energy', 'Materials', 'Industrials', 'Consumer Discretionary',
        'Consumer Staples', 'Health Care', 'Financials',
        'Information Technology', 'Communication Services', 'Utilities', 'Real Estate'
    ]
})

print(f"GICS mapping: {len(gics_map)} sectors")
print(f"  Merged rows with gsector: {merged['gsector'].notna().sum():,}")

merged = merged.merge(gics_map, on='gsector', how='left')
print(f"After merge: {len(merged):,} rows")
print(f"  Rows with sector_name: {merged['sector_name'].notna().sum():,}")

inspect_dataframe(merged, "Final Merged Dataset")


## Step 11: Prepare Analysis Data


In [ ]:
# ============================================================================

print_section("STEP 11: Prepare Analysis Data")

analysis_data = merged[[
    'gvkey', 'permno', 'date', 'datadate', 'conm', 'tic',
    'sector_name', 'ret_crsp', 'mktcap', 'ni', 'xrd', 'prc_crsp'
]].copy()

analysis_data = analysis_data.rename(columns={
    'datadate': 'report_date',
    'conm': 'company_name',
    'tic': 'ticker',
    'sector_name': 'sector',
    'ni': 'net_income_loss',
    'xrd': 'research_and_development_expense'
})

print("Renamed columns:")
print(f"  {list(analysis_data.columns)}")

# Create research_not indicator
print("\nCreating research_not indicator...")
print(f"  Rows with xrd (research): {analysis_data['research_and_development_expense'].notna().sum():,}")
print(f"  Rows without xrd: {analysis_data['research_and_development_expense'].isna().sum():,}")

analysis_data['research_not'] = analysis_data['research_and_development_expense'].apply(
    lambda x: 'Without_XRD' if pd.isna(x) else 'With_XRD'
)

print(f"\nresearch_not distribution:")
print(analysis_data['research_not'].value_counts())

inspect_dataframe(analysis_data, "Analysis Data (Final)")


## Step 12: Data Quality Check for Portfolio Calculation


In [ ]:
# ============================================================================

print_section("STEP 12: Data Quality Check for Portfolio Calculation")

print("Checking required columns for portfolio calculation...")
required_cols = ['ret_crsp', 'prc_crsp', 'mktcap']
for col in required_cols:
    if col in analysis_data.columns:
        non_null = analysis_data[col].notna().sum()
        null = analysis_data[col].isna().sum()
        print(f"  {col}: {non_null:,} non-null, {null:,} null")
    else:
        print(f"  {col}: MISSING COLUMN!")

# Check for zero returns
print(f"\nZero returns: {(analysis_data['ret_crsp'] == 0).sum():,}")
print(f"Non-zero returns: {(analysis_data['ret_crsp'] != 0).sum():,}")

# Check for valid prices
print(f"\nValid prices (> 0): {(analysis_data['prc_crsp'] > 0).sum():,}")
print(f"Zero or negative prices: {(analysis_data['prc_crsp'] <= 0).sum():,}")

# Check for valid market cap
print(f"\nValid market cap (> 0): {(analysis_data['mktcap'] > 0).sum():,}")
print(f"Zero or negative market cap: {(analysis_data['mktcap'] <= 0).sum():,}")

# Filter for portfolio calculation requirements
portfolio_ready = analysis_data[
    analysis_data['ret_crsp'].notna() & 
    analysis_data['prc_crsp'].notna() & 
    analysis_data['mktcap'].notna() & 
    (analysis_data['ret_crsp'] != 0)
].copy()

print(f"\nRows ready for portfolio calculation: {len(portfolio_ready):,} (from {len(analysis_data):,})")

# Check date range
if 'date' in portfolio_ready.columns:
    dates = pd.to_datetime(portfolio_ready['date'], errors='coerce')
    valid_dates = dates.dropna()
    if len(valid_dates) > 0:
        print(f"\nDate range in portfolio-ready data: {valid_dates.min()} to {valid_dates.max()}")

print_section("EDA COMPLETE")
print("You can now inspect the data at each step to understand the data flow.")
print("Use this information to debug issues in the main script.")
